In [2]:
import pandas as pd

In [3]:
pred = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/prod_data_invoice_detection_test_sample_21_05_2026_v1_qwen32B.csv")
# pred = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/invoice_detection_results_13_05_2026_v2_qwen32B_startendtokens.csv")

In [4]:
import re
import json
import pandas as pd
import numpy as np

def extract_first_json(text):
    text = (
        text
        .replace(" ","")
        .replace("\n","")
        .replace("True", "true")
        .replace("False", "false")
        .replace("None", "null")
        .replace("'",'"')
    )
    match = re.search(r'\{.*?\}', text)
    if match:
        try:
            json_data = json.loads(match.group())
            return json_data
        except json.JSONDecodeError as e:
            print(match.group())
            print("failed", e)
            return {}
    else:
        print("format failed")
        return {}

def extract_last_json(text):
    text = (
        text
        .replace(" ","")
        .replace("\n","")
        .replace("True", "true")
        .replace("False", "false")
        .replace("None", "null")
        .replace("'",'"')
    )
    # Find all JSON objects in the text
    matches = list(re.finditer(r'\{.*?\}', text))
    if matches:
        # Get the last match
        last_match = matches[-1]
        try:
            json_data = json.loads(last_match.group())
            return json_data
        except json.JSONDecodeError as e:
            print(last_match.group())
            print("failed", e)
            return {}
    else:
        print("format failed")
        return {}
    
def parse_date_safe(date):
    try:
        return pd.to_datetime(date)
    except:
        return np.nan

In [5]:
pred["pred_parsed"] = pred["pred_llm"].apply(extract_last_json)

In [6]:
pred["pred_is_invoice_inside"] = pred["pred_parsed"].apply(lambda x: x.get("is_invoice_inside", False))
pred["pred_start_page"] = pred["pred_parsed"].apply(lambda x: x.get("start_page", False))
pred["pred_end_page"] = pred["pred_parsed"].apply(lambda x: x.get("end_page", False))

In [7]:
pred

,id,ticket_uuid,attachment_id,zendesk_id,comment_id,status,file_name,file_extension,status_written_at,created_at,...,number_of_pages,clean_text,text,local_file_path,lentgh,pred_llm,pred_parsed,pred_is_invoice_inside,pred_start_page,pred_end_page
0,7606776,cedd31a9-3880-5bd7-bd60-b4caa2ae3ea0,60052381,NaN,NaN,processed,60052381_DR_II_422_26_aus_Schreibmaschine.PDF,pdf,2026-04-17 10:44:14,2026-04-17 10:44:14,...,1,<page_1>\nobergerichtsvollzieherin\namtsgerich...,Obergerichtsvollzieherin\nAmtsgericht Wolfsbur...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,2788,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
1,7606784,1a8b97e7-e92b-586b-abea-3491d8729e03,60052682,NaN,NaN,processed,60052682_DR-II_121925_Nachr_1te_Mahnung_der_Ko...,pdf,2026-04-17 10:44:16,2026-04-17 10:44:16,...,1,<page_1>\nbörsenstr. 44\n26382 wilhelmshaven\n...,Börsenstr. 44\n26382 Wilhelmshaven\nTel. 04421...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,1915,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
2,7606824,d95492cc-1bb0-513b-8030-585e968ef7fe,60052437,NaN,NaN,processed,60052437_Prot_58926_17042026_081036.pdf,pdf,2026-04-17 10:44:21,2026-04-17 10:44:21,...,2,<page_1>\ngabriele kruse\nobergerichtsvollzieh...,Gabriele Kruse\nObergerichtsvollzieherin\nKöni...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,3827,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 2, '...",True,2,2
3,7606844,d95492cc-1bb0-513b-8030-585e968ef7fe,60052435,NaN,NaN,processed,60052435_Dokument_58926_17042026_081030.pdf,pdf,2026-04-17 10:44:27,2026-04-17 10:44:27,...,1,"<page_1>\ngabriele kruse\nkönigstraße 11, zi e...","Gabriele Kruse\nKönigstraße 11, Zi EG 005\nObe...",/Users/melih.gorgulu/Desktop/Projects/aftercou...,2657,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
4,7606852,9dff4528-65d8-558a-8aa1-596261076401,60052743,NaN,NaN,processed,60052743_1_DR_331_26_An_Gl_Nachricht_HBAntrag.PDF,pdf,2026-04-17 10:44:28,2026-04-17 10:44:28,...,8,<page_1>\nhauptgerichtsvollzieher\nnutzweg 3\n...,Hauptgerichtsvollzieher\nNutzweg 3\nPaul Bäuer...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,10568,"Okay, let me tackle this problem step by step....","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,252,f99b4405-13de-58c4-8e92-4c809e98fde8,15617654-1,15617654.0,NaN,processed,20250114_162601.jpg,jpg,2025-01-25 01:16:20,2025-01-24 14:13:35,...,1,<page_1>\nüberweisung - übernahmebestätigung\n...,überweisung - übernahmebestätigung\n09.10.2024...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,403,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1
796,253,fa3b139c-b4d0-5eab-84de-b567286d8f10,15617657-1,15617657.0,NaN,processed,image.png,png,2025-01-25 01:16:20,2025-01-24 14:14:24,...,1,"<page_1>\nseptember 4, 2024\n17,85 € additiona...","September 4, 2024\n17,85 € Additional revenue:...",/Users/melih.gorgulu/Desktop/Projects/aftercou...,2915,"Okay, let me try to figure this out. The user ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1
797,254,1f465535-7442-5b08-a468-fc22d94b131a,15617658-1,15617658.0,NaN,processed,1000020244.jpg,jpg,2025-01-25 01:16:20,2025-01-24 14:14:36,...,1,<page_1>\n13.01./13.01.\nsepa überweisung\n- 8...,"13.01./13.01.\nSEPA Überweisung\n- 86,19 2\nKL...",/Users/melih.gorgulu/Desktop/Projects/aftercou...,83,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1
798,255,10c0096f-0dd7-5433-bf53-0da3cdd9f0b9,15617660-1,15617660.0,NaN,processed,IMG_5858.jpeg,jpeg,2025-01-25 01:16:20,2025-01-24 14:14:43,...,1,<page_1>\nrevolut\nconfirmare transfer\ngenera...,Revolut\nConfirmare t

In [8]:
from __future__ import annotations

import re
from typing import Optional, Tuple


_SEITE_PATTERNS = [
    r"(?i)\bseite\s*:?\s*\d+",
    r"(?i)\bseite\s*\d+\s*/\s*\d+",
    r"(?i)\bseite\s*\d+\s*von\s*\d+",
    r"(?i)\bblatt\s*:?\s*\d+",
]
_SEITE_REGEXES = [re.compile(p) for p in _SEITE_PATTERNS]

_PAGE_TAG_RE = re.compile(r"<page_(\d+)>")
_DIGIT_RE = re.compile(r"\d+")

_GUARD_RE = re.compile(
    r"(?im)^\s*(?:protokoll"
    r"|verm[oö0]gens\s*verzeichnis"
    r"|verm[oö0]gens\s*auskunft(?:s\s*protokoll|protokoll)?"
    r"|dritt\s*ausk[uü]nfte"
    r"|ergebnis(?:se)?(?:\s+der\s+verm[oö0]gens\s*auskunft)?"
    r")\b"
)

DEFAULT_SHORT_PAGE_WORD_THRESHOLD = 100


def _check_seite_marker_inside(text: str) -> Tuple[bool, Optional[str]]:
    for regex in _SEITE_REGEXES:
        match = regex.search(text)
        if match:
            return True, match.group()
    return False, None


def _extract_page_text(full_text: str, page_idx: int, number_of_pages: int) -> str:
    if page_idx == number_of_pages:
        start_marker = f"<page_{page_idx}>"
        if start_marker in full_text:
            return full_text.split(start_marker)[-1]
        return ""

    start_marker = f"<page_{page_idx}>"
    end_marker = f"<page_{page_idx + 1}>"
    if start_marker in full_text and end_marker in full_text:
        return full_text.split(start_marker)[-1].split(end_marker)[0]
    return ""


def _count_pages(text: str) -> int:
    nums = [int(n) for n in _PAGE_TAG_RE.findall(text)]
    return max(nums) if nums else 0


def _arg_find_consecutive_sequence(found_page_markers: list[str]) -> list[int]:
    if not found_page_markers:
        return []

    first_number_match = _DIGIT_RE.search(found_page_markers[0])
    if not first_number_match:
        return []
    first_number = int(first_number_match.group())
    first_pattern = _DIGIT_RE.sub("", found_page_markers[0]).strip().lower()

    for i in range(1, len(found_page_markers)):
        current_marker = found_page_markers[i]
        current_pattern = _DIGIT_RE.sub("", current_marker).strip().lower()
        if current_pattern != first_pattern:
            return list(range(i))
        current_number_match = _DIGIT_RE.search(current_marker)
        if not current_number_match:
            return list(range(i))
        if int(current_number_match.group()) != first_number + i:
            return list(range(i))

    return list(range(len(found_page_markers)))


def _extend_by_page_markers(start_page: int, end_page: int, text: str) -> Tuple[int, int]:
    n_of_pages = _count_pages(text)
    invoice_page_text = _extract_page_text(text, start_page, n_of_pages)

    cur_text = invoice_page_text
    cur_page_idx = start_page
    found_page_markers: list[str] = []
    found_page_markers_page_indices: list[int] = []

    while True:
        is_with_marker, matched = _check_seite_marker_inside(cur_text)
        if not (is_with_marker and matched):
            break
        found_page_markers.append(matched)
        found_page_markers_page_indices.append(cur_page_idx)
        cur_page_idx += 1
        cur_text = _extract_page_text(text, cur_page_idx, n_of_pages)

    indexes = _arg_find_consecutive_sequence(found_page_markers)
    if indexes:
        end_page = found_page_markers_page_indices[indexes[-1]]

    is_with_marker, matched_start_marker = _check_seite_marker_inside(invoice_page_text)
    matched_number = _DIGIT_RE.search(matched_start_marker) if matched_start_marker else None

    if (
        is_with_marker
        and matched_start_marker
        and matched_number
        and int(matched_number.group()) > 1
    ):
        start_pattern = _DIGIT_RE.sub("", matched_start_marker).strip().lower()
        expected_number = int(matched_number.group()) - 1
        prev_page_idx = start_page - 1

        while prev_page_idx >= 1 and expected_number > 0:
            prev_page_text = _extract_page_text(text, prev_page_idx, n_of_pages)
            is_with_prev_marker, matched_prev = _check_seite_marker_inside(prev_page_text)
            if not (is_with_prev_marker and matched_prev):
                break

            prev_pattern = _DIGIT_RE.sub("", matched_prev).strip().lower()
            if prev_pattern != start_pattern:
                break

            prev_number_match = _DIGIT_RE.search(matched_prev)
            if not prev_number_match:
                break
            if int(prev_number_match.group()) != expected_number:
                break

            start_page = prev_page_idx
            expected_number -= 1
            prev_page_idx -= 1

    return start_page, end_page


def _extend_if_next_page_short(
    start_page: int,
    end_page: int,
    text: str,
    word_threshold: int = DEFAULT_SHORT_PAGE_WORD_THRESHOLD,
) -> int:
    if start_page != end_page:
        return end_page

    n_of_pages = _count_pages(text)
    next_page_idx = start_page + 1
    next_page_text = _extract_page_text(text, next_page_idx, n_of_pages)

    if not next_page_text:
        return end_page
    if len(next_page_text.split()) >= word_threshold:
        return end_page
    if _GUARD_RE.search(next_page_text):
        return end_page

    return next_page_idx


def correct_invoice_pages(
    model_output: Optional[dict],
    text_with_page_markers: Optional[str],
    short_page_word_threshold: int = DEFAULT_SHORT_PAGE_WORD_THRESHOLD,
) -> dict:
    if not model_output or not text_with_page_markers:
        return model_output or {}

    if not bool(model_output.get("is_invoice_inside")):
        return model_output

    try:
        start_page = int(model_output.get("start_page"))
        end_page = int(model_output.get("end_page"))
    except (TypeError, ValueError):
        return model_output
    if start_page < 1 or end_page < 1:
        return model_output

    new_start, new_end = _extend_by_page_markers(start_page, end_page, text_with_page_markers)
    if new_end == end_page:
        new_end = _extend_if_next_page_short(
            new_start, new_end, text_with_page_markers,
            word_threshold=short_page_word_threshold,
        )

    if new_start == start_page and new_end == end_page:
        return model_output

    corrected = dict(model_output)
    corrected["start_page"] = new_start
    corrected["end_page"] = new_end
    return corrected


In [ ]:
def _apply_correction(row):
    model_output = row["pred_parsed"] if isinstance(row["pred_parsed"], dict) else {}
    corrected = correct_invoice_pages(model_output, row.get("clean_text"))
    return pd.Series({
        "pred_start_page_corrected": corrected.get("start_page", row["pred_start_page"]),
        "pred_end_page_corrected": corrected.get("end_page", row["pred_end_page"]),
    })

pred[["pred_start_page_corrected", "pred_end_page_corrected"]] = pred.apply(_apply_correction, axis=1)


In [10]:
pred

,id,ticket_uuid,attachment_id,zendesk_id,comment_id,status,file_name,file_extension,status_written_at,created_at,...,text,local_file_path,lentgh,pred_llm,pred_parsed,pred_is_invoice_inside,pred_start_page,pred_end_page,pred_start_page_corrected,pred_end_page_corrected
0,7606776,cedd31a9-3880-5bd7-bd60-b4caa2ae3ea0,60052381,NaN,NaN,processed,60052381_DR_II_422_26_aus_Schreibmaschine.PDF,pdf,2026-04-17 10:44:14,2026-04-17 10:44:14,...,Obergerichtsvollzieherin\nAmtsgericht Wolfsbur...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,2788,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
1,7606784,1a8b97e7-e92b-586b-abea-3491d8729e03,60052682,NaN,NaN,processed,60052682_DR-II_121925_Nachr_1te_Mahnung_der_Ko...,pdf,2026-04-17 10:44:16,2026-04-17 10:44:16,...,Börsenstr. 44\n26382 Wilhelmshaven\nTel. 04421...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,1915,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
2,7606824,d95492cc-1bb0-513b-8030-585e968ef7fe,60052437,NaN,NaN,processed,60052437_Prot_58926_17042026_081036.pdf,pdf,2026-04-17 10:44:21,2026-04-17 10:44:21,...,Gabriele Kruse\nObergerichtsvollzieherin\nKöni...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,3827,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 2, '...",True,2,2,2,2
3,7606844,d95492cc-1bb0-513b-8030-585e968ef7fe,60052435,NaN,NaN,processed,60052435_Dokument_58926_17042026_081030.pdf,pdf,2026-04-17 10:44:27,2026-04-17 10:44:27,...,"Gabriele Kruse\nKönigstraße 11, Zi EG 005\nObe...",/Users/melih.gorgulu/Desktop/Projects/aftercou...,2657,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
4,7606852,9dff4528-65d8-558a-8aa1-596261076401,60052743,NaN,NaN,processed,60052743_1_DR_331_26_An_Gl_Nachricht_HBAntrag.PDF,pdf,2026-04-17 10:44:28,2026-04-17 10:44:28,...,Hauptgerichtsvollzieher\nNutzweg 3\nPaul Bäuer...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,10568,"Okay, let me tackle this problem step by step....","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,252,f99b4405-13de-58c4-8e92-4c809e98fde8,15617654-1,15617654.0,NaN,processed,20250114_162601.jpg,jpg,2025-01-25 01:16:20,2025-01-24 14:13:35,...,überweisung - übernahmebestätigung\n09.10.2024...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,403,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1
796,253,fa3b139c-b4d0-5eab-84de-b567286d8f10,15617657-1,15617657.0,NaN,processed,image.png,png,2025-01-25 01:16:20,2025-01-24 14:14:24,...,"September 4, 2024\n17,85 € Additional revenue:...",/Users/melih.gorgulu/Desktop/Projects/aftercou...,2915,"Okay, let me try to figure this out. The user ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1
797,254,1f465535-7442-5b08-a468-fc22d94b131a,15617658-1,15617658.0,NaN,processed,1000020244.jpg,jpg,2025-01-25 01:16:20,2025-01-24 14:14:36,...,"13.01./13.01.\nSEPA Überweisung\n- 86,19 2\nKL...",/Users/melih.gorgulu/Desktop/Projects/aftercou...,83,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1
798,255,10c0096f-0dd7-5433-bf53-0da3cdd9f0b9,15617660-1,15617660.0,NaN,processed,IMG_5858.jpeg,jpeg,2025-01-25 01:16:20,2025-01-24 14:14:43,...,Revolut\nConfirmare transfer\nGenerat la data ...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,1479,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1


In [12]:
pred.pred_is_invoice_inside.value_counts()

pred_is_invoice_inside
True     410
False    390
Name: count, dtype: int64

In [14]:
# import sys
# from pathlib import Path
# import dill

# INTENT_RECOGNITION_PATH = Path("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/intent_recognition")
# if str(INTENT_RECOGNITION_PATH) not in sys.path:
#     sys.path.insert(0, str(INTENT_RECOGNITION_PATH))

# from src.services.models.aftercourt_tokenizer import ClassificationSpacyLemmaTokenizer  # noqa: F401

# VEC_PATH = Path("/Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court/models/vectorizers/05-12-2026_tf_idf_vectorizer_v1.dill")
# RF_MODEL_PATH = Path("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/models/classification/15-05-2026_binary_va_rf_classifier_v1.dill")

# with open(VEC_PATH, "rb") as f:
#     ve_tfidf = dill.load(f)
# with open(RF_MODEL_PATH, "rb") as f:
#     ve_clf = dill.load(f)

# ve_input_text = pred["clean_text"].fillna(pred["text"]).fillna("").astype(str)

# # remove page markers from clean_text (<page_1>, <page_2>, etc.)
# ve_input_text = ve_input_text.apply(lambda x: re.sub(r"<page_\d+>", "", x))

# ve_features = ve_tfidf.transform(ve_input_text)
# pred["ve_pred"] = ve_clf.predict(ve_features)

In [18]:
pred.drop(columns=["ve_pred"], inplace=True)

In [ ]:
#pred.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/prod_data_invoice_detection_test_sample_21_05_2026_v1_qwen32B_corrected.csv", index=False)

In [38]:
import pandas as pd
pred = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/prod_data_invoice_detection_test_sample_21_05_2026_v1_qwen32B_corrected.csv")

In [39]:
pred

,id,ticket_uuid,attachment_id,zendesk_id,comment_id,status,file_name,file_extension,status_written_at,created_at,...,text,local_file_path,lentgh,pred_llm,pred_parsed,pred_is_invoice_inside,pred_start_page,pred_end_page,pred_start_page_corrected,pred_end_page_corrected
0,7606776,cedd31a9-3880-5bd7-bd60-b4caa2ae3ea0,60052381,NaN,NaN,processed,60052381_DR_II_422_26_aus_Schreibmaschine.PDF,pdf,2026-04-17 10:44:14,2026-04-17 10:44:14,...,Obergerichtsvollzieherin\nAmtsgericht Wolfsbur...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,2788,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
1,7606784,1a8b97e7-e92b-586b-abea-3491d8729e03,60052682,NaN,NaN,processed,60052682_DR-II_121925_Nachr_1te_Mahnung_der_Ko...,pdf,2026-04-17 10:44:16,2026-04-17 10:44:16,...,Börsenstr. 44\n26382 Wilhelmshaven\nTel. 04421...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,1915,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
2,7606824,d95492cc-1bb0-513b-8030-585e968ef7fe,60052437,NaN,NaN,processed,60052437_Prot_58926_17042026_081036.pdf,pdf,2026-04-17 10:44:21,2026-04-17 10:44:21,...,Gabriele Kruse\nObergerichtsvollzieherin\nKöni...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,3827,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 2, '...",True,2,2,2,2
3,7606844,d95492cc-1bb0-513b-8030-585e968ef7fe,60052435,NaN,NaN,processed,60052435_Dokument_58926_17042026_081030.pdf,pdf,2026-04-17 10:44:27,2026-04-17 10:44:27,...,"Gabriele Kruse\nKönigstraße 11, Zi EG 005\nObe...",/Users/melih.gorgulu/Desktop/Projects/aftercou...,2657,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,1
4,7606852,9dff4528-65d8-558a-8aa1-596261076401,60052743,NaN,NaN,processed,60052743_1_DR_331_26_An_Gl_Nachricht_HBAntrag.PDF,pdf,2026-04-17 10:44:28,2026-04-17 10:44:28,...,Hauptgerichtsvollzieher\nNutzweg 3\nPaul Bäuer...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,10568,"Okay, let me tackle this problem step by step....","{'is_invoice_inside': True, 'start_page': 1, '...",True,1,1,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,252,f99b4405-13de-58c4-8e92-4c809e98fde8,15617654-1,15617654.0,NaN,processed,20250114_162601.jpg,jpg,2025-01-25 01:16:20,2025-01-24 14:13:35,...,überweisung - übernahmebestätigung\n09.10.2024...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,403,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1
796,253,fa3b139c-b4d0-5eab-84de-b567286d8f10,15617657-1,15617657.0,NaN,processed,image.png,png,2025-01-25 01:16:20,2025-01-24 14:14:24,...,"September 4, 2024\n17,85 € Additional revenue:...",/Users/melih.gorgulu/Desktop/Projects/aftercou...,2915,"Okay, let me try to figure this out. The user ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1
797,254,1f465535-7442-5b08-a468-fc22d94b131a,15617658-1,15617658.0,NaN,processed,1000020244.jpg,jpg,2025-01-25 01:16:20,2025-01-24 14:14:36,...,"13.01./13.01.\nSEPA Überweisung\n- 86,19 2\nKL...",/Users/melih.gorgulu/Desktop/Projects/aftercou...,83,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1
798,255,10c0096f-0dd7-5433-bf53-0da3cdd9f0b9,15617660-1,15617660.0,NaN,processed,IMG_5858.jpeg,jpeg,2025-01-25 01:16:20,2025-01-24 14:14:43,...,Revolut\nConfirmare transfer\nGenerat la data ...,/Users/melih.gorgulu/Desktop/Projects/aftercou...,1479,"Okay, let's tackle this problem step by step. ...","{'is_invoice_inside': False, 'start_page': -1,...",False,-1,-1,-1,-1


In [25]:
pred.columns

Index(['id', 'ticket_uuid', 'attachment_id', 'zendesk_id', 'comment_id',
       'status', 'file_name', 'file_extension', 'status_written_at',
       'created_at', 's3_key', 's3_bucket', 'job_id', 's3_link', 'updated_at',
       'coming_from', 'number_of_pages', 'clean_text', 'text',
       'local_file_path', 'lentgh', 'pred_llm', 'pred_parsed',
       'pred_is_invoice_inside', 'pred_start_page', 'pred_end_page',
       'pred_start_page_corrected', 'pred_end_page_corrected'],
      dtype='object')

In [32]:
# --- Debug correction algorithm for attachment_id 60057057 ---
DEBUG_ATTACHMENT_ID = 60057057

_row_mask = pred["attachment_id"].astype(str) == str(DEBUG_ATTACHMENT_ID)
assert _row_mask.any(), f"attachment_id {DEBUG_ATTACHMENT_ID} not found in pred"
_row = pred.loc[_row_mask].iloc[0]

_model_output = _row["pred_parsed"] if isinstance(_row["pred_parsed"], dict) else {}
_text = _row.get("clean_text") or ""
_n_pages_total = _count_pages(_text)

print("=" * 80)
print(f"attachment_id           : {_row['attachment_id']}")
print(f"number_of_pages (col)   : {_row.get('number_of_pages')}")
print(f"number_of_pages (text)  : {_n_pages_total}")
print(f"pred_parsed             : {_model_output}")
print(f"pred_start_page         : {_row.get('pred_start_page')}")
print(f"pred_end_page           : {_row.get('pred_end_page')}")
print(f"pred_start_page_corr.   : {_row.get('pred_start_page_corrected')}")
print(f"pred_end_page_corr.     : {_row.get('pred_end_page_corrected')}")
print("=" * 80)

# Re-run the correction with verbose tracing
_start_page = int(_model_output.get("start_page"))
_end_page = int(_model_output.get("end_page"))

print("\n--- Per-page Seite/Blatt marker scan ---")
for _p in range(1, _n_pages_total + 1):
    _ptxt = _extract_page_text(_text, _p, _n_pages_total)
    _has, _matched = _check_seite_marker_inside(_ptxt)
    _wc = len(_ptxt.split())
    _guard = bool(_GUARD_RE.search(_ptxt))
    _flag = "<-- start" if _p == _start_page else ("<-- end" if _p == _end_page else "")
    print(
        f"  page {_p:>2}: words={_wc:>5}  marker={_matched!r:<35}  "
        f"guard_hit={_guard}  {_flag}"
    )

print("\n--- _extend_by_page_markers trace ---")
_inv_text = _extract_page_text(_text, _start_page, _n_pages_total)
_cur_text = _inv_text
_cur_idx = _start_page
_found, _found_idx = [], []
while True:
    _has, _m = _check_seite_marker_inside(_cur_text)
    if not (_has and _m):
        print(f"  page {_cur_idx}: no marker -> stop")
        break
    print(f"  page {_cur_idx}: marker={_m!r}")
    _found.append(_m)
    _found_idx.append(_cur_idx)
    _cur_idx += 1
    _cur_text = _extract_page_text(_text, _cur_idx, _n_pages_total)

_seq = _arg_find_consecutive_sequence(_found)
print(f"  consecutive markers indices: {_seq} -> pages {[_found_idx[i] for i in _seq]}")

_new_start, _new_end = _extend_by_page_markers(_start_page, _end_page, _text)
print(f"  after marker extend: start={_new_start}, end={_new_end}")

if _new_end == _end_page:
    _new_end2 = _extend_if_next_page_short(
        _new_start, _new_end, _text, DEFAULT_SHORT_PAGE_WORD_THRESHOLD
    )
    print(f"  after short-next-page extend: end={_new_end2}")
    _next_idx = _new_start + 1
    _next_text = _extract_page_text(_text, _next_idx, _n_pages_total)
    print(
        f"    next page (idx={_next_idx}) words={len(_next_text.split())}, "
        f"guard_hit={bool(_GUARD_RE.search(_next_text))}"
    )

print("\n--- Final corrected output ---")
print(correct_invoice_pages(_model_output, _text))


attachment_id           : 60057057
number_of_pages (col)   : 2
number_of_pages (text)  : 2
pred_parsed             : {'is_invoice_inside': True, 'start_page': 1, 'end_page': 1}
pred_start_page         : 1
pred_end_page           : 1
pred_start_page_corr.   : 1
pred_end_page_corr.     : 1

--- Per-page Seite/Blatt marker scan ---
  page  1: words=  448  marker='seite 1'                            guard_hit=True  <-- start
  page  2: words=    5  marker='seite 2'                            guard_hit=False  

--- _extend_by_page_markers trace ---
  page 1: marker='seite 1'
  page 2: marker='seite 2'
  page 3: no marker -> stop
  consecutive markers indices: [0, 1] -> pages [1, 2]
  after marker extend: start=1, end=2

--- Final corrected output ---
{'is_invoice_inside': True, 'start_page': 1, 'end_page': 2}


In [35]:
# --- Debug correction algorithm for attachment_id 60059370 ---
DEBUG_ATTACHMENT_ID = 60059370

_row_mask = pred["attachment_id"].astype(str) == str(DEBUG_ATTACHMENT_ID)
assert _row_mask.any(), f"attachment_id {DEBUG_ATTACHMENT_ID} not found in pred"
_row = pred.loc[_row_mask].iloc[0]

_model_output = _row["pred_parsed"] if isinstance(_row["pred_parsed"], dict) else {}
_text = _row.get("clean_text") or ""
_n_pages_total = _count_pages(_text)

print("=" * 80)
print(f"attachment_id           : {_row['attachment_id']}")
print(f"number_of_pages (col)   : {_row.get('number_of_pages')}")
print(f"number_of_pages (text)  : {_n_pages_total}")
print(f"pred_parsed             : {_model_output}")
print(f"pred_start_page         : {_row.get('pred_start_page')}")
print(f"pred_end_page           : {_row.get('pred_end_page')}")
print(f"pred_start_page_corr.   : {_row.get('pred_start_page_corrected')}")
print(f"pred_end_page_corr.     : {_row.get('pred_end_page_corrected')}")
print("=" * 80)

# Re-run the correction with verbose tracing
_start_page = int(_model_output.get("start_page"))
_end_page = int(_model_output.get("end_page"))

print("\n--- Per-page Seite/Blatt marker scan ---")
for _p in range(1, _n_pages_total + 1):
    _ptxt = _extract_page_text(_text, _p, _n_pages_total)
    _has, _matched = _check_seite_marker_inside(_ptxt)
    _wc = len(_ptxt.split())
    _guard = bool(_GUARD_RE.search(_ptxt))
    _flag = "<-- start" if _p == _start_page else ("<-- end" if _p == _end_page else "")
    print(
        f"  page {_p:>2}: words={_wc:>5}  marker={_matched!r:<35}  "
        f"guard_hit={_guard}  {_flag}"
    )

print("\n--- _extend_by_page_markers trace ---")
_inv_text = _extract_page_text(_text, _start_page, _n_pages_total)
_cur_text = _inv_text
_cur_idx = _start_page
_found, _found_idx = [], []
while True:
    _has, _m = _check_seite_marker_inside(_cur_text)
    if not (_has and _m):
        print(f"  page {_cur_idx}: no marker -> stop")
        break
    print(f"  page {_cur_idx}: marker={_m!r}")
    _found.append(_m)
    _found_idx.append(_cur_idx)
    _cur_idx += 1
    _cur_text = _extract_page_text(_text, _cur_idx, _n_pages_total)

_seq = _arg_find_consecutive_sequence(_found)
print(f"  consecutive markers indices: {_seq} -> pages {[_found_idx[i] for i in _seq]}")

_new_start, _new_end = _extend_by_page_markers(_start_page, _end_page, _text)
print(f"  after marker extend: start={_new_start}, end={_new_end}")

if _new_end == _end_page:
    _new_end2 = _extend_if_next_page_short(
        _new_start, _new_end, _text, DEFAULT_SHORT_PAGE_WORD_THRESHOLD
    )
    print(f"  after short-next-page extend: end={_new_end2}")
    _next_idx = _new_start + 1
    _next_text = _extract_page_text(_text, _next_idx, _n_pages_total)
    print(
        f"    next page (idx={_next_idx}) words={len(_next_text.split())}, "
        f"guard_hit={bool(_GUARD_RE.search(_next_text))}"
    )

print("\n--- Final corrected output ---")
print(correct_invoice_pages(_model_output, _text))


attachment_id           : 60059370
number_of_pages (col)   : 4
number_of_pages (text)  : 4
pred_parsed             : {'is_invoice_inside': True, 'start_page': 1, 'end_page': 3}
pred_start_page         : 1
pred_end_page           : 3
pred_start_page_corr.   : 1
pred_end_page_corr.     : 1

--- Per-page Seite/Blatt marker scan ---
  page  1: words=  343  marker='seite 2'                            guard_hit=False  <-- start
  page  2: words=  304  marker='seite 1'                            guard_hit=False  
  page  3: words=   86  marker='seite 2'                            guard_hit=False  <-- end
  page  4: words=  131  marker='seite\n1'                           guard_hit=True  

--- _extend_by_page_markers trace ---
  page 1: marker='seite 2'
  page 2: marker='seite 1'
  page 3: marker='seite 2'
  page 4: marker='seite\n1'
  page 5: no marker -> stop
  consecutive markers indices: [0] -> pages [1]
  after marker extend: start=1, end=1

--- Final corrected output ---
{'is_invoice_ins

# Feature: Add new page if cant find a slug

In [49]:
from pathlib import Path
import sys
INTENT_RECOGNITION_PATH = Path("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/intent_recognition")
if str(INTENT_RECOGNITION_PATH) not in sys.path:
    sys.path.insert(0, str(INTENT_RECOGNITION_PATH))

from src.services.attachment_processing.aftercourt_extractors.ladung.slug_extractor import SlugExtractor


slug_extractor = SlugExtractor()

def is_slug_found(full_text, start_page, end_page):
    for p in range(start_page, end_page + 1):
        page_text = _extract_page_text(full_text, p, _n_pages_total)
        if slug_extractor.extract(page_text):
            return True
    return False

In [55]:
def add_page_if_slug_found_in_margin(pred_row):
    start_page = pred_row["pred_start_page_corrected"]
    end_page = pred_row["pred_end_page_corrected"]
    
    if start_page < 1 or end_page < 1:
        return start_page, end_page
    
    full_text = pred_row.get("clean_text") or ""
    slug_found = is_slug_found(full_text, start_page, end_page)
    n_of_pages = _count_pages(full_text)
    if slug_found:
        return start_page, end_page
    else:
        check_start = max(1, start_page - 1)
        
        if is_slug_found(full_text, check_start, check_start):
            return check_start, end_page
        
        check_end = min(end_page + 1, n_of_pages)
        if is_slug_found(full_text, check_end, check_end):
            return start_page, check_end

    return start_page, end_page

In [56]:
pred_slug = pred.apply(add_page_if_slug_found_in_margin, axis=1, result_type="expand")
pred[["pred_start_page_final", "pred_end_page_final"]] = pred_slug

In [57]:
# Rows where the slug step changed start/end vs. the corrected values
_changed_mask = (
    (pred["pred_start_page_final"] != pred["pred_start_page_corrected"])
    | (pred["pred_end_page_final"] != pred["pred_end_page_corrected"])
)
_changed = pred.loc[
    _changed_mask,
    [
        "attachment_id",
        "number_of_pages",
        "pred_start_page_corrected",
        "pred_end_page_corrected",
        "pred_start_page_final",
        "pred_end_page_final",
    ],
].copy()
_changed["start_delta"] = _changed["pred_start_page_final"] - _changed["pred_start_page_corrected"]
_changed["end_delta"] = _changed["pred_end_page_final"] - _changed["pred_end_page_corrected"]

print(f"Slug step changed {len(_changed)} / {len(pred)} rows")
_changed


Slug step changed 30 / 800 rows


,attachment_id,number_of_pages,pred_start_page_corrected,pred_end_page_corrected,pred_start_page_final,pred_end_page_final,start_delta,end_delta
2,60052437,2,2,2,1,2,-1,0
13,60059595,2,2,2,1,2,-1,0
23,60056982,2,2,2,1,2,-1,0
33,60057125,2,2,2,1,2,-1,0
36,60059370,4,1,1,1,2,0,1
39,60057026,2,2,2,1,2,-1,0
53,60062467,3,2,3,1,3,-1,0
82,60062291,3,2,3,1,3,-1,0
107,60063753,3,2,3,1,3,-1,0
131,60065564,3,2,3,1,3,-1,0


In [ ]:
#pred.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/prod_data_invoice_detection_test_sample_21_05_2026_v1_qwen32B_corrected_viaslug.csv", index=False)

# if detected invoice contains guard re, set it as not invoice

In [59]:
pred.columns

Index(['id', 'ticket_uuid', 'attachment_id', 'zendesk_id', 'comment_id',
       'status', 'file_name', 'file_extension', 'status_written_at',
       'created_at', 's3_key', 's3_bucket', 'job_id', 's3_link', 'updated_at',
       'coming_from', 'number_of_pages', 'clean_text', 'text',
       'local_file_path', 'lentgh', 'pred_llm', 'pred_parsed',
       'pred_is_invoice_inside', 'pred_start_page', 'pred_end_page',
       'pred_start_page_corrected', 'pred_end_page_corrected', 'slug_found',
       'pred_start_page_final', 'pred_end_page_final'],
      dtype='object')

In [75]:
def get_invoice_pages(row):
    return list(range(row["pred_start_page_final"], row["pred_end_page_final"] + 1))


def extract_invoice_text(row):
    full_text = row.get("clean_text") or ""
    pages = get_invoice_pages(row)
    page_texts = [_extract_page_text(full_text, p, _count_pages(full_text)) for p in pages]
    return "\n".join(page_texts)

_GUARD_PROTOCOL_PATTERNS = [
    re.compile(r"(?i)\bprotokoll\b"),
    re.compile(
        r"(?i)\b(?:[uü]ber|zur)\s+abgabe\s+(?:der|die)\s+verm[oö0]gens\s*auskunft\b"
    ),
    re.compile(
        r"(?i)\b(?:forderung|befriedigung)\s+des\s+gl[aä]ubigers\b"
    ),
]

_GUARD_PROTOCOL_MIN_MATCHES = 2


def contains_guard_protocol(text):
    hits = sum(1 for p in _GUARD_PROTOCOL_PATTERNS if p.search(text))
    return hits >= _GUARD_PROTOCOL_MIN_MATCHES


def filter_protokol_docs(pred_row):
    invoice_text = extract_invoice_text(pred_row)
    return contains_guard_protocol(invoice_text)
    




In [76]:
pred['protokoll_guard'] = pred.apply(filter_protokol_docs, axis=1)

In [ ]:
# pred.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/prod_data_invoice_detection_test_sample_21_05_2026_v1_qwen32B_corrected_viaslug_and_protokol.csv", index=False)

Note: With this basic filtering we filter some useful documents as well. Check this later.